# Lab 3: Quantization and Predictive Coding (2026 Version)

In this lab, you will study two key ideas in video compression:

- **quantization**
- **predictive coding**

The lab uses two video frames. Instead of coding the second frame directly, you will code the **difference** between the two frames. This difference is then quantized, entropy-coded with Huffman coding, decoded, and used to reconstruct the second frame.

You are expected to complete all **TODO** parts. You are **allowed** to use the codes from previous labs.

This lab is designed to connect:
- frame differencing
- quantization
- entropy
- Huffman coding
- reconstruction quality

## Group Information

**Group number:**  
**Member 1:**  
**Member 2:**  
**Member 3:**

# Introduction

The goal of this lab is to

- get familiar with how to perform a quantization process
- learn how to do video compression through the predictive coding technique

In this lab, you will:

1. extract frame number 80 and 85 from a video
2. convert both frames from RGB to YUV
3. keep only the **Y layers**
4. compute the **frame difference**
5. quantize the frame difference
6. compute its entropy
7. perform Huffman coding on the quantized difference
8. reconstruct frame 85 from frame 80 and the coded frame difference
9. evaluate the quality of the reconstructed frame using PSNR

This lab illustrates one of the central ideas in video coding:

> instead of coding a frame directly, we predict it from a previous frame and code only the difference

## Learning Objectives

After completing this lab, you should be able to:

- extract two selected frames from a video file
- convert RGB frames to YUV and isolate the Y channel
- compute and quantize a frame difference
- explain why a frame difference may be easier to compress than a full frame
- compute the entropy of the quantized difference
- build and use a Huffman codebook for the frame difference
- reconstruct a frame from a reference frame and a coded residual
- evaluate reconstruction quality with PSNR

## Setup

Before starting, place the video file in the same folder as this notebook.

Then run the following cell to import the required libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import heapq

## Part 1: Read the Video and Extract Frame 80 and Frame 85

In this first part, read the video file and extract frame number 80 and frame number 85.

### Task
- Open the video file
- Extract frame 80 and frame 85
- Convert them from BGR to RGB
- Display both frames

In [ ]:
def generate_fallback_frame(index, height=240, width=320):
    """Generate a synthetic RGB frame for fallback use."""
    x = np.linspace(0, 255, width, dtype=np.float32)
    y = np.linspace(0, 255, height, dtype=np.float32)
    X, Y = np.meshgrid(x, y)

    shift = index * 2.0
    R = np.mod(X + shift, 256)
    G = np.mod(Y + 0.5 * shift, 256)
    B = np.mod(0.5 * X + 0.5 * Y + shift, 256)

    frame = np.stack([R, G, B], axis=2)
    return np.clip(frame, 0, 255).astype(np.uint8)

def extract_video_frames(video_name, indices):
    cap = cv2.VideoCapture(video_name)

    if not cap.isOpened():
        print("Could not open the video file. Using fallback synthetic frames.")
        return {idx: generate_fallback_frame(idx) for idx in indices}

    frames = {}

    # TODO: read each requested frame index
    for idx in indices:
        ...
        ...
        ...
        ...

    cap.release()
    return frames

video_name = "test video.mp4"
frames = extract_video_frames(video_name, [80, 85])

frame80_rgb = frames[80]
frame85_rgb = frames[85]

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.imshow(frame80_rgb)
plt.title("RGB Frame 80")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(frame85_rgb)
plt.title("RGB Frame 85")
plt.axis("off")

plt.tight_layout()
plt.show()

## Part 2: Convert RGB to YUV

Convert both RGB frames to YUV and keep only the **Y** layers.

The Y channel is computed as:

\[
Y = 0.299R + 0.587G + 0.114B
\]

In the rest of the lab, the Y layers will be treated as grayscale frames.

In [ ]:
def frameRGB2YUV(RGBframe):
    RGBframe = RGBframe.astype(np.float32)

    # TODO: extract R, G, B
    R = ...
    G = ...
    B = ...

    # TODO: compute Y, U, V
    Y = ...
    U = ...
    V = ...

    # TODO: stack Y, U, V into one array
    YUVframe = ...
    return YUVframe

YUV80 = frameRGB2YUV(frame80_rgb)
YUV85 = frameRGB2YUV(frame85_rgb)

Y80 = np.clip(YUV80[:, :, 0], 0, 255).astype(np.uint8)
Y85 = np.clip(YUV85[:, :, 0], 0, 255).astype(np.uint8)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.imshow(Y80, cmap="gray")
plt.title("Y Frame 80")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(Y85, cmap="gray")
plt.title("Y Frame 85")
plt.axis("off")

plt.tight_layout()
plt.show()

\## Part 3: Differential Video Coding

Compute the frame difference between frame 85 and frame 80:

\[
d = \text{frame}_{85} - \text{frame}_{80}
\]

Then quantize the frame difference with a quantization step size of 4.

### Quantization rule
Use

\[
d_q = \left\lfloor \frac{d}{4} \right\rfloor
\]\

The inverse quantization step will be applied later during reconstruction by multiplying by 4.

In [ ]:
def Diffimage(frame1, frame2):
    """Compute quantized frame difference d_q = floor((frame2-frame1)/4)."""
    frame1 = frame1.astype(np.float32)
    frame2 = frame2.astype(np.float32)

    # TODO: compute frame difference
    diff = ...

    # TODO: quantize with step size 4
    quantized_diff = ...
    return quantized_diff

diff_q = Diffimage(Y80, Y85)

# For visualization only: shift values to visible range
diff_vis = diff_q.astype(np.float32)
diff_vis = diff_vis - diff_vis.min()
if diff_vis.max() > 0:
    diff_vis = diff_vis / diff_vis.max() * 255.0
diff_vis = diff_vis.astype(np.uint8)

plt.figure(figsize=(6, 4))
plt.imshow(diff_vis, cmap="gray")
plt.title("Quantized Frame Difference (visualized)")
plt.axis("off")
plt.show()

print("Quantized difference min:", diff_q.min())
print("Quantized difference max:", diff_q.max())

NameError: name 'Y80' is not defined

## Part 4: Entropy of the Quantized Frame Difference

Compute the probability distribution and entropy of the quantized frame difference.

Because the frame difference contains signed values, use a dictionary of symbol counts rather than a 256-bin grayscale histogram.

The entropy is

\[
H = -\sum_x p(x)\log_2 p(x)
\]

In [ ]:
def Calcent(input_image):
    """Calculate entropy of an image with possibly signed integer values and plot its probability distribution."""
    # TODO: compute unique values and their counts
    values, counts = ...

    # TODO: convert counts to probabilities
    probabilities = ...

    # TODO: compute entropy
    entropy = ...

    # TODO: build a probability dictionary p
    p = ...

    plt.figure(figsize=(10, 4))
    plt.bar(values, probabilities, width=0.8)
    plt.title("Probability Distribution of the Quantized Frame Difference")
    plt.xlabel("Quantized Difference Value")
    plt.ylabel("Probability")
    plt.grid(True, alpha=0.3)
    plt.show()

    return entropy, p

entropy_diff, p = Calcent(diff_q)
print("Entropy of the quantized frame difference:", entropy_diff, "bits/pixel")

## Part 5: Huffman Coding

Construct a Huffman codebook for the quantized frame difference and use it to encode and decode the frame difference.

In [ ]:
def Huffmandict(p):
    """Generate a Huffman code dictionary from a probability dictionary."""
    heap = [[prob, [symbol, ""]] for symbol, prob in p.items()]
    heapq.heapify(heap)

    if len(heap) == 0:
        return {}
    if len(heap) == 1:
        symbol = heap[0][1][0]
        return {symbol: "0"}

    # TODO: repeatedly merge the two least probable nodes
    while len(heap) > 1:
        lo = ...
        hi = ...

        # TODO: prepend 0 and 1 to the two branches
        for pair in lo[1:]:
            ...
        for pair in hi[1:]:
            ...

        # TODO: push merged node back into the heap
        ...

    huff_list = sorted(heap[0][1:], key=lambda pair: (len(pair[1]), str(pair[0])))
    huff_dict = {symbol: codeword for symbol, codeword in huff_list}
    return huff_dict

Huffdict = Huffmandict(p)

print("Number of coded symbols:", len(Huffdict))
for i, (symbol, codeword) in enumerate(list(Huffdict.items())[:10]):
    print(symbol, "->", codeword)

In [ ]:
def Huffmanenco(OriginalImage, Huffdict):
    """Perform Huffman encoding on an integer image and return a bitstream string."""
    flat = OriginalImage.flatten()

    # TODO: replace each symbol by its Huffman codeword
    encoded = ...
    return encoded

In [ ]:
def Huffmandeco(EncodedImage, Huffdict, shape):
    """Decode a Huffman bitstream back into an integer image."""
    reverse_dict = {v: k for k, v in Huffdict.items()}

    decoded_symbols = []
    buffer = ""

    for bit in EncodedImage:
        buffer += bit

        # TODO: when a valid codeword is found, append the symbol and reset the buffer
        if ...:
            ...
            ...

    # TODO: convert to array and reshape
    decoded = ...
    return decoded

## Part 6: Information Rate and Coding Efficiency

After encoding the quantized frame difference, compute:

- the **information rate**

\[
R = \frac{\text{number of coded bits}}{\text{number of pixels}}
\]

- the **entropy coding efficiency**

\[
\text{Efficiency} = \frac{H}{R}\times 100\%
\]

A higher efficiency means the practical coding rate is closer to the entropy.

In [ ]:
encoded_diff = Huffmanenco(diff_q, Huffdict)
decoded_diff = Huffmandeco(encoded_diff, Huffdict, diff_q.shape)

# TODO: compute information rate
rate_value = ...

# TODO: compute coding efficiency
efficiency_value = ...

print("Encoded bitstream length:", len(encoded_diff), "bits")
print("Information rate R:", rate_value, "bits/pixel")
print("Entropy H:", entropy_diff, "bits/pixel")
print("Entropy coding efficiency:", efficiency_value, "%")
print("Decoded difference correct:", np.array_equal(diff_q, decoded_diff))

## Part 7: Reconstruction

Reconstruct frame 85 by first performing inverse quantization on the frame difference and then adding it to frame 80.

Inverse quantization:

\[
d \approx 4 d_q
\]

Reconstruction:

\[
\widehat{\text{frame}}_{85} = \text{frame}_{80} + 4d_q
\]

In [ ]:
def Reconsimage(frame, framedifference):
    """Reconstruct a frame using the previous frame and inverse-quantized difference."""
    frame = frame.astype(np.int16)
    framedifference = framedifference.astype(np.int16)

    # TODO: inverse quantization
    inv_quantized = ...

    # TODO: reconstruct frame and clip to valid range
    reconstructed = ...
    reconstructed = ...
    return reconstructed

Y85_reconstructed = Reconsimage(Y80, decoded_diff)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.imshow(Y85, cmap="gray")
plt.title("Original Y Frame 85")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(Y85_reconstructed, cmap="gray")
plt.title("Reconstructed Y Frame 85")
plt.axis("off")

plt.tight_layout()
plt.show()

## Part 8: PSNR

Compute the Peak Signal-to-Noise Ratio (PSNR) between the original frame 85 and the reconstructed frame 85.

\[
\text{PSNR} = 10\log_{10}\left(\frac{255^2}{\text{MSE}}\right)
\]

In [ ]:
def CalculatePSNR(im1, im2):
    if im1.shape != im2.shape:
        raise ValueError("Images must have the same shape.")

    # TODO: convert to float
    im1 = ...
    im2 = ...

    # TODO: compute mean squared error
    mse = ...

    if mse == 0:
        return float("inf")

    # TODO: compute PSNR
    psnr = ...
    return psnr

psnr_value = CalculatePSNR(Y85, Y85_reconstructed)
print("PSNR of reconstructed Y frame 85:", psnr_value, "dB")

## Part 9: Difference Image Between Original and Reconstructed Frame 85

Visualize the reconstruction error.

In [ ]:
recon_diff = np.abs(Y85.astype(np.int16) - Y85_reconstructed.astype(np.int16)).astype(np.uint8)

plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
plt.imshow(Y85, cmap="gray")
plt.title("Original Y Frame 85")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(Y85_reconstructed, cmap="gray")
plt.title("Reconstructed Y Frame 85")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(recon_diff, cmap="gray")
plt.title("Absolute Reconstruction Error")
plt.axis("off")

plt.tight_layout()
plt.show()

# Report

Your report should include:

- Y frame 80 and 85 side by side
- the image of the quantized frame difference
- the probability distribution of the frame difference
- the entropy of the quantized frame difference
- the information rate \(R\)
- the entropy coding efficiency
- frame 85 side by side with its reconstructed version
- the PSNR of the reconstructed Y frame 85

## Discussion questions

1. Why is the frame difference often easier to compress than the full frame?
2. Which stage is lossy, and which stage is lossless?
3. Why does quantization lower reconstruction quality?
4. Why does Huffman coding not itself introduce distortion?
5. How would the entropy change if the two frames were very different?